In [2]:
# install packages
%pip install -q nltk transformers scikit-learn matplotlib torch numpy

# Import libraries
import nltk
import numpy as np
import torch

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import cross_val_score
from transformers import BertModel, BertTokenizer

# Download NLTK treebank dataset
nltk.download('treebank', quiet=True)
from nltk.corpus import treebank

Note: you may need to restart the kernel to use updated packages.


In [3]:
# each sentence is a list of (word, pos_tag) tuples
sentences = treebank.tagged_sents()

# first sentence from the treebank corpus
first_sentence = sentences[0]
print(first_sentence)

# separate the words and the corresponding POS tags
words = []
pos_order = []
for word, pos in first_sentence:
    words.append(word)
    pos_order.append(pos)

# printing sentence and order of POS tags --> visualization of how the data looks like
print("Sentence:", " ".join(words))
print("POS Order:", " ".join(pos_order))

[('Pierre', 'NNP'), ('Vinken', 'NNP'), (',', ','), ('61', 'CD'), ('years', 'NNS'), ('old', 'JJ'), (',', ','), ('will', 'MD'), ('join', 'VB'), ('the', 'DT'), ('board', 'NN'), ('as', 'IN'), ('a', 'DT'), ('nonexecutive', 'JJ'), ('director', 'NN'), ('Nov.', 'NNP'), ('29', 'CD'), ('.', '.')]
Sentence: Pierre Vinken , 61 years old , will join the board as a nonexecutive director Nov. 29 .
POS Order: NNP NNP , CD NNS JJ , MD VB DT NN IN DT JJ NN NNP CD .


In [4]:
# Create sentence-level data
sentences_text = [" ".join([word for word, _ in sentence]) for sentence in sentences]
sentence_lengths = [len(sentence) for sentence in sentences]

# printing to visualize the structure
print("Sample sentence length data:")
for i in range(3):  # Show first 3 examples
    print(f"Sentence {i+1}: {sentences_text[i]}")
    print(f"Length: {sentence_lengths[i]}")
    print()

Sample sentence length data:
Sentence 1: Pierre Vinken , 61 years old , will join the board as a nonexecutive director Nov. 29 .
Length: 18

Sentence 2: Mr. Vinken is chairman of Elsevier N.V. , the Dutch publishing group .
Length: 13

Sentence 3: Rudolph Agnew , 55 years old and former chairman of Consolidated Gold Fields PLC , was named *-1 a nonexecutive director of this British industrial conglomerate .
Length: 27



In [5]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# create BERT embeddings for each sentence
def get_bert_embeddings(texts):
    embeddings = []
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    model.eval()
    
    # Process in batches to avoid memory issues
    batch_size = 32
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors='pt').to(device)
            outputs = model(**inputs)
            
            # Get token embedding (first token) for each sentence
            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.extend(batch_embeddings)
    
    return np.array(embeddings)

# Get BERT embeddings for sentences
print("Generating BERT embeddings for sentences...")
X_sentence_features = get_bert_embeddings(sentences_text)
print(f"Sentence BERT embeddings shape: {X_sentence_features.shape}")

Generating BERT embeddings for sentences...
Sentence BERT embeddings shape: (3914, 768)


In [6]:
def train_random_forest_model(features, labels):
    """
    Train random forest regression model to predict sentence length.
    
    Parameters:
      features (np.ndarray): Feature matrix of shape (num_samples, num_features).
      labels (np.ndarray): Sentence length labels.
      
    Returns:
      model: Trained random forest regression model.
      metrics: Dictionary containing evaluation metrics.
      X_test: Test features.
      y_test: Test labels.
    """
    # splitting into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(
        features, labels, test_size=0.2, random_state=42
    )
    
    # Random Forest regression model
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Predictions on the test set
    predictions = model.predict(X_test)
    
    # evaluation
    metrics = {
        "mean_squared_error": mean_squared_error(y_test, predictions),
        "mean_absolute_error": mean_absolute_error(y_test, predictions),
        "r2_score": r2_score(y_test, predictions),
    }
    
    return model, metrics, X_test, y_test

In [7]:
# Define features and labels
features = X_sentence_features
labels = sentence_lengths

# Train the Random Forest model using the function
rf_model, metrics, X_test, y_test = train_random_forest_model(features, labels)

# Get predictions for later evaluation
rf_preds = rf_model.predict(X_test)

# Display the metrics from the training function
print("Random Forest Model Performance:")
for metric_name, metric_value in metrics.items():
    print(f"- {metric_name}: {metric_value:.4f}")

KeyboardInterrupt: 

In [ ]:
import pandas as pd
# Collect all results
results = []
results.append(evaluate_model(y_test, rf_preds, 'Random Forest'))

# Display results as a DataFrame
results_df = pd.DataFrame(results)
display(results_df) 


,Model,MSE,RMSE,MAE,R2
0,Random Forest,61.599799,7.848554,5.954342,0.577612


In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Display predictions and actual values for sentences
def display_prediction(index):
    if index < 0 or index >= len(X_test):
        print("Index out of range. Please select a valid index.")
        return
    
    # Get the prediction and actual value
    prediction = rf_model.predict([X_test[index]])[0]
    actual = y_test[index]
    
    # Try to display the original sentence text if available
    try:
        test_indices = range(len(X_test))
        original_index = list(test_indices)[index]
        original_sentence = sentences_text[original_index]
        print(f"Original Sentence: {original_sentence}")
    except:
        print("Original sentence text not available")
    
    print(f"Sentence Index: {index}")
    print(f"Predicted Sentence Length: {prediction:.2f}")
    print(f"Actual Sentence Length: {actual}")
    print(f"Error: {abs(prediction - actual):.2f}")


slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(X_test) - 1,
    step=1,
    description='Sentence Index:',
    continuous_update=False
)


interactive_widget = widgets.interactive(display_prediction, index=slider)

display(interactive_widget)

interactive(children=(IntSlider(value=0, continuous_update=False, description='Sentence Index:', max=782), Out…